# 01 — Build paper-brief evaluation corpus

Freeze usable `full_text_plain` from **archived local Papers** into `data/paper_brief_evaluation/corpus/`.

**Prerequisite:** each DOI already has a `Paper` row in the app Postgres (`just up`). This notebook does not archive. Run it with `just notebooks`, not `just sandbox`.

Domain calls: `get_paper_by_doi`, `usable_full_text_plain`, and (when the body is missing) `inform_source_record` then `inform_full_text` with `force=True`. Do not import `paper_reviewer.flows`. This notebook does not write `PaperBrief`.

**Git:** corpus files (`.txt` + `manifest.jsonl`) and later `{run_id}/` results under `data/paper_brief_evaluation/` are tracked so you can commit them. They stay out of the production image (`.dockerignore`).

In [ ]:
# Curated eval set. Must match archived Paper.doi in local Postgres.
# Strip and case are normalized below.
DOI_LIST = [
    # "10.1000/EXAMPLE",
]

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from paper_reviewer.db import create_db_engine, create_session_factory, session_scope
from paper_reviewer.ingest.pubmed.pmc_cloud import usable_full_text_plain
from paper_reviewer.models.paper import get_paper_by_doi
from paper_reviewer.topic_scope.fulfill_papers_metadata.inform import (
    inform_full_text,
    inform_source_record,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
CORPUS_DIR = REPO_ROOT / "data" / "paper_brief_evaluation" / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.jsonl"
SESSION_FACTORY = create_session_factory(create_db_engine())
print(f"repo root: {REPO_ROOT}")
print(f"corpus dir: {CORPUS_DIR}")

In [ ]:
def normalize_doi(raw: str) -> str:
    return raw.strip().upper()


def corpus_filename(doi: str) -> str:
    return f"{doi.replace('/', '_')}.txt"


def unique_dois(raw_dois: list[str]) -> list[str]:
    seen: dict[str, None] = {}
    for raw in raw_dois:
        doi = normalize_doi(raw)
        if doi:
            seen.setdefault(doi, None)
    return list(seen)


def first_filename_collision(dois: list[str]) -> tuple[str, str] | None:
    by_name: dict[str, str] = {}
    for doi in dois:
        name = corpus_filename(doi)
        owner = by_name.get(name)
        if owner is not None and owner != doi:
            return owner, doi
        by_name[name] = doi
    return None

In [ ]:
dois = unique_dois(DOI_LIST)
collision = first_filename_collision(dois)
if collision is not None:
    left, right = collision
    raise RuntimeError(
        "Corpus filenames collide for "
        f"{left!r} and {right!r} -> {corpus_filename(left)}. "
        "Step 1 stopped. Previous corpus files were not changed."
    )

accepted: list[dict] = []
skipped_missing: list[str] = []
skipped_no_text: list[str] = []
errors: list[tuple[str, str]] = []

for doi in dois:
    try:
        with session_scope(SESSION_FACTORY) as session:
            paper = get_paper_by_doi(session, doi)
            if paper is None:
                print(f"SKIP {doi}: no Paper row (this notebook does not archive)")
                skipped_missing.append(doi)
                continue
            paper_id = paper.id
            body = usable_full_text_plain(paper.full_text_plain)
            title = paper.title
            journal = paper.journal
            published_year = paper.published_year

        if body is None:
            print(
                f"FETCH {doi}: no usable full text; "
                "inform_source_record then inform_full_text (force=True)"
            )
            inform_source_record(paper_id, force=True)
            inform_full_text(paper_id, force=True)
            with session_scope(SESSION_FACTORY) as session:
                paper = get_paper_by_doi(session, doi)
                if paper is None:
                    print(f"SKIP {doi}: Paper missing after inform")
                    skipped_missing.append(doi)
                    continue
                body = usable_full_text_plain(paper.full_text_plain)
                title = paper.title
                journal = paper.journal
                published_year = paper.published_year
            if body is None:
                print(f"SKIP {doi}: source cannot supply usable full text")
                skipped_no_text.append(doi)
                continue

        filename = corpus_filename(doi)
        accepted.append(
            {
                "doi": doi,
                "title": title,
                "journal": journal,
                "published_year": published_year,
                "filename": filename,
                "_body": body,
            }
        )
        print(f"ACCEPT {doi} -> {filename}")
    except Exception as exc:
        print(f"ERROR {doi}: {exc}")
        errors.append((doi, str(exc)))

if not accepted:
    print("No accepted papers. No corpus files written.")
else:
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    for row in accepted:
        (CORPUS_DIR / row["filename"]).write_text(row["_body"], encoding="utf-8")
    with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
        for row in accepted:
            record = {
                "doi": row["doi"],
                "title": row["title"],
                "journal": row["journal"],
                "published_year": row["published_year"],
                "filename": row["filename"],
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Wrote {len(accepted)} .txt file(s) and {MANIFEST_PATH}")

print("---")
print(f"accepted: {len(accepted)}")
print(f"skipped (no Paper): {len(skipped_missing)}")
print(f"skipped (no usable text): {len(skipped_no_text)}")
print(f"errors: {len(errors)}")

After a successful build, commit the corpus (and later `{run_id}/` results) if you want them in the repository:

```bash
git add data/paper_brief_evaluation/
```

Production images still exclude `data/` via `.dockerignore`.